# Slide aggregation — the five missing PAIP slides

Turns the per-patch features into one tensor per slide, for the five slides:

`training_data_19_nonMSIH` · `training_data_30_MSIH` · `training_data_41_nonMSIH` · `training_data_42_MSIH` · `training_data_46_nonMSIH`

Three methods, all writing into `paip_data\slide_aggregation` next to the 73 slides already there:

| Method | What it does | Output shape | Folder |
|---|---|---|---|
| Averaging | average every patch into one vector | `(1, D)` | `Averaging\<model>` |
| Tissue type clustering | classify each crop into 9 CRC100K tissue types, average per type | `(9, D)` | `Tissue_Type_Clustering\<model>` |
| Caption based | label each patch with CONCH, average per label | `(14, D)` and `(15, D)` | `Caption_based_aggregation[_15_classes]\<model>` |

Run `Feature_Extraction_Missing_PAIP_fivecrop.ipynb` first — all three read the `.pt` files it writes.

**Caption based needs one extra step.** The other two only need features, but caption based needs a predicted
class for every patch, and `classification_results_paip.csv` only covers the 73 slides. Section 3a runs CONCH
over the five slides' patches to produce those labels before section 3b aggregates them.

In [7]:
import os
import glob
import json
import shutil
from datetime import datetime

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm

PAIP_ROOT     = r"D:\Aamir Gulzar\KSA_project2\paip_data"
FEATURES_ROOT = os.path.join(PAIP_ROOT, "Features")
AGG_ROOT      = os.path.join(PAIP_ROOT, "slide_aggregation")
PATCH_DATA_DIR = os.path.join(PAIP_ROOT, "patch_data")
NON_WHITE_CSV  = os.path.join(PAIP_ROOT, "final_merged_without_white.csv")

REPO_ROOT   = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier"
TTC_MODELS  = os.path.join(REPO_ROOT, "TissueClassifier_CRC100K", "models")
CAPTION_DIR = os.path.join(REPO_ROOT, "slide_aggregation", "caption_generation")

SLIDES = [
    "training_data_19_nonMSIH",
    "training_data_30_MSIH",
    "training_data_41_nonMSIH",
    "training_data_42_MSIH",
    "training_data_46_nonMSIH",
]

# feature folder, embedding size, and the tissue classifier trained on that model
MODELS = {
    "ConchV1":     {"features": "Conch_v1_fivecrop_Updated",    "dim": 512,  "ttc": "ann_crc100k_ConchV1.pth"},
    "Conch1_5":    {"features": "Conch_v1_5_fivecrop_Updated",  "dim": 768,  "ttc": "ann_crc100k_conch15.pth"},
    "UNI2":        {"features": "Uni_2h_fivecrop_Updated",      "dim": 1536, "ttc": "ann_crc100k_uni2.pth"},
    "Virchow2":    {"features": "Virchow2_fivecrop_Updated",    "dim": 2560, "ttc": "ann_crc100k_Virchow2.pth"},
    "H-Optimus-1": {"features": "H_Optimus_1_fivecrop_Updated", "dim": 1536, "ttc": "ann_crc100k_Hoptimus.pth"},
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


def feature_dir(model, slide):
    return os.path.join(FEATURES_ROOT, MODELS[model]["features"], slide)


# check the features are actually there before doing anything
print(f"\n{'model':<14}" + "".join(f"{s.split('_data_')[1]:>10}" for s in SLIDES))
for model in MODELS:
    counts = [len(glob.glob(os.path.join(feature_dir(model, s), "*.pt"))) for s in SLIDES]
    print(f"{model:<14}" + "".join(f"{c:>10,}" for c in counts))

device: cuda

model         19_nonMSIH   30_MSIH41_nonMSIH   42_MSIH46_nonMSIH
ConchV1              898     1,466       170     1,116       569
Conch1_5             898     1,466       170     1,116       569
UNI2                 898     1,466       170     1,116       569
Virchow2             898     1,466       170     1,116       569
H-Optimus-1          898     1,466       170     1,116       569


---
## 1. Averaging

For each slide: average the 5 crops of every patch, then average all the patches. One vector per slide.

In [2]:
def average_slide(model, slide):
    """Every patch of the slide -> one (1, D) vector."""
    pt_files = glob.glob(os.path.join(feature_dir(model, slide), "*.pt"))
    if not pt_files:
        return None

    per_patch = []
    for f in pt_files:
        t = torch.load(f, map_location="cpu", weights_only=True)   # (5, D)
        per_patch.append(t.mean(dim=0))                            # (D,)

    return torch.stack(per_patch).mean(dim=0, keepdim=True)        # (1, D)


for model in MODELS:
    out_dir = os.path.join(AGG_ROOT, "Averaging", model)
    os.makedirs(out_dir, exist_ok=True)

    for slide in SLIDES:
        result = average_slide(model, slide)
        if result is None:
            print(f"  {model} / {slide}: no features, skipping")
            continue

        torch.save(result, os.path.join(out_dir, slide + ".pt"))
        print(f"  {model:<14} {slide:<26} {tuple(result.shape)}")

  ConchV1        training_data_19_nonMSIH   (1, 512)
  ConchV1        training_data_30_MSIH      (1, 512)
  ConchV1        training_data_41_nonMSIH   (1, 512)
  ConchV1        training_data_42_MSIH      (1, 512)
  ConchV1        training_data_46_nonMSIH   (1, 512)
  Conch1_5       training_data_19_nonMSIH   (1, 768)
  Conch1_5       training_data_30_MSIH      (1, 768)
  Conch1_5       training_data_41_nonMSIH   (1, 768)
  Conch1_5       training_data_42_MSIH      (1, 768)
  Conch1_5       training_data_46_nonMSIH   (1, 768)
  UNI2           training_data_19_nonMSIH   (1, 1536)
  UNI2           training_data_30_MSIH      (1, 1536)
  UNI2           training_data_41_nonMSIH   (1, 1536)
  UNI2           training_data_42_MSIH      (1, 1536)
  UNI2           training_data_46_nonMSIH   (1, 1536)
  Virchow2       training_data_19_nonMSIH   (1, 2560)
  Virchow2       training_data_30_MSIH      (1, 2560)
  Virchow2       training_data_41_nonMSIH   (1, 2560)
  Virchow2       training_data_42_MSIH

---
## 2. Tissue type clustering

Each crop goes through a small ANN trained on CRC100K that sorts it into one of 9 tissue types. The crops
belonging to each type are then averaged, giving one row per tissue type. Types that don't appear in a slide
stay as zeros.

Note this classifies **every crop separately** (5 per patch), not the averaged patch vector — same as the
existing scripts.

In [3]:
NUM_CLASSES = 9
BATCH_SIZE = 512


def tissue_type_slide(model_name, slide, ann):
    """All crops -> predicted tissue type -> average per type. Returns (9, D)."""
    cfg = MODELS[model_name]
    pt_files = glob.glob(os.path.join(feature_dir(model_name, slide), "*.pt"))
    if not pt_files:
        return None

    crops = []
    for f in pt_files:
        t = torch.load(f, map_location="cpu", weights_only=True)   # (5, D)
        if t.ndim == 1:
            t = t.unsqueeze(0)
        crops.append(t)

    crops = torch.cat(crops).float()          # (N*5, D)

    preds = []
    with torch.no_grad():
        for i in range(0, len(crops), BATCH_SIZE):
            batch = crops[i:i + BATCH_SIZE].to(device)
            preds.append(ann(batch).argmax(dim=1).cpu())
    preds = torch.cat(preds)                  # (N*5,)

    out = torch.zeros(NUM_CLASSES, cfg["dim"])
    for cls in range(NUM_CLASSES):
        mask = preds == cls
        if mask.any():
            out[cls] = crops[mask].mean(dim=0)

    return out, preds


for model_name, cfg in MODELS.items():
    ann_path = os.path.join(TTC_MODELS, cfg["ttc"])
    print(f"\n{model_name}  <-  {cfg['ttc']}")

    # these were saved as the whole module, not a state_dict
    ann = torch.load(ann_path, map_location=device, weights_only=False)
    ann.to(device).eval()

    out_dir = os.path.join(AGG_ROOT, "Tissue_Type_Clustering", model_name)
    os.makedirs(out_dir, exist_ok=True)

    for slide in SLIDES:
        result = tissue_type_slide(model_name, slide, ann)
        if result is None:
            print(f"  {slide}: no features, skipping")
            continue

        tensor, preds = result
        used = int((tensor.abs().sum(dim=1) > 0).sum())
        torch.save(tensor, os.path.join(out_dir, slide + ".pt"))
        print(f"  {slide:<26} {tuple(tensor.shape)}  {used}/9 tissue types present")

    del ann
    if device.type == "cuda":
        torch.cuda.empty_cache()


ConchV1  <-  ann_crc100k_ConchV1.pth
  training_data_19_nonMSIH   (9, 512)  9/9 tissue types present
  training_data_30_MSIH      (9, 512)  9/9 tissue types present
  training_data_41_nonMSIH   (9, 512)  7/9 tissue types present
  training_data_42_MSIH      (9, 512)  9/9 tissue types present
  training_data_46_nonMSIH   (9, 512)  9/9 tissue types present

Conch1_5  <-  ann_crc100k_conch15.pth
  training_data_19_nonMSIH   (9, 768)  9/9 tissue types present
  training_data_30_MSIH      (9, 768)  9/9 tissue types present
  training_data_41_nonMSIH   (9, 768)  8/9 tissue types present
  training_data_42_MSIH      (9, 768)  9/9 tissue types present
  training_data_46_nonMSIH   (9, 768)  9/9 tissue types present

UNI2  <-  ann_crc100k_uni2.pth
  training_data_19_nonMSIH   (9, 1536)  9/9 tissue types present
  training_data_30_MSIH      (9, 1536)  9/9 tissue types present
  training_data_41_nonMSIH   (9, 1536)  6/9 tissue types present
  training_data_42_MSIH      (9, 1536)  9/9 tissue types

---
## 3. Caption based

### 3a. Label the patches with CONCH

`classification_results_paip.csv` has a row per patch — patch name, predicted class, score — but only for the
73 slides. This runs the same zero-shot CONCH classifier over the five slides' patches and appends their rows.

Two label sets, because both aggregations exist: the 14-class one and the 15-class one (which adds TIL and
BACK). The class order comes from the prompt config files rather than being typed out here, so it can't drift
out of step with the weights.

In [3]:
# `conch` is not pip-installed - it is vendored inside caption_generation/, and the
# original zero-shot notebook works because it runs from that folder. This notebook
# sits one level up, so put that folder on the path before importing.
import sys

if CAPTION_DIR not in sys.path:
    sys.path.insert(0, CAPTION_DIR)

from conch.open_clip_custom import create_model_from_pretrained

CONCH_CKPT = os.path.join(CAPTION_DIR, "checkpoints", "CONCH", "pytorch_model.bin")

LABEL_SETS = {
    14: {
        "weights": os.path.join(CAPTION_DIR, "classifier_weights", "Conch_zeroshot_weights.pt"),
        "config" : os.path.join(CAPTION_DIR, "config_files", "Custom_prompts_all_per_class.json"),
        "csv"    : os.path.join(CAPTION_DIR, "Results", "classification_results_paip.csv"),
        "agg_dir": "Caption_based_aggregation",
    },
    15: {
        "weights": os.path.join(CAPTION_DIR, "classifier_weights", "Conch_zeroshot_weights_15_classes.pt"),
        "config" : os.path.join(CAPTION_DIR, "config_files", "config_15_classes.json"),
        "csv"    : os.path.join(CAPTION_DIR, "Results", "classification_results_paip_15_classes.csv"),
        "agg_dir": "Caption_based_aggregation_15_classes",
    },
}

# class order = the order of classnames in the prompt config, which is what
# built the zero-shot weights in the first place
for n, cfg in LABEL_SETS.items():
    with open(cfg["config"]) as f:
        classnames = json.load(f)["0"]["classnames"]
    cfg["classes"] = list(classnames.keys())
    print(f"{n}-class order: {cfg['classes']}")

conch, preprocess = create_model_from_pretrained(
    model_cfg="conch_ViT-B-16",
    checkpoint_path=CONCH_CKPT,
    device=device,
    force_image_size=224,
)
conch.eval()
print("\nCONCH loaded")

c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\models\layers\__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


14-class order: ['ADI', 'DEB', 'LYM', 'PLC', 'LYA', 'MUC', 'MUS', 'NORM', 'ADE', 'STR', 'MES', 'CAR', 'PDC', 'SIG']
15-class order: ['ADI', 'DEB', 'TIL', 'PLC', 'LYA', 'LYM', 'MUS', 'MUC', 'NORM', 'ADE', 'STR', 'CAR', 'PDC', 'SIG', 'BACK']

CONCH loaded


In [4]:
CONCH_BATCH = 64

valid_names = set(pd.read_csv(NON_WHITE_CSV)["patch_name"].astype(str))


def label_patches(paths, weights, classes):
    """Zero-shot classify a list of patch images. Returns (labels, scores)."""
    labels, scores = [], []

    with torch.inference_mode():
        for i in tqdm(range(0, len(paths), CONCH_BATCH), desc="  classifying", leave=False):
            batch_paths = paths[i:i + CONCH_BATCH]
            batch = torch.stack([
                preprocess(Image.open(p).convert("RGB")) for p in batch_paths
            ]).to(device)

            feats = conch.encode_image(batch)          # (B, 512)
            sims = feats @ weights                     # (B, n_classes)
            top_score, top_idx = sims.max(dim=1)

            labels.extend(classes[j] for j in top_idx.cpu().tolist())
            scores.extend(top_score.cpu().tolist())

    return labels, scores


for n_classes, cfg in LABEL_SETS.items():
    print(f"\n=== {n_classes}-class labels -> {os.path.basename(cfg['csv'])}")
    weights = torch.load(cfg["weights"], map_location=device)

    existing = pd.read_csv(cfg["csv"])
    already = set(existing["patch_name"].astype(str))

    rows = []
    for slide in SLIDES:
        paths = [p for p in glob.glob(os.path.join(PATCH_DATA_DIR, slide, "*.png"))
                 if os.path.basename(p) in valid_names
                 and os.path.basename(p) not in already]

        if not paths:
            print(f"  {slide}: already labelled")
            continue

        print(f"  {slide}: {len(paths):,} patches")
        labels, scores = label_patches(paths, weights, cfg["classes"])
        rows += [{"patch_name": os.path.basename(p), "label": l, "score": s}
                 for p, l, s in zip(paths, labels, scores)]

    if not rows:
        print("  nothing new to add")
        continue

    backup = f"{cfg['csv']}.bak-{datetime.now():%Y%m%d-%H%M%S}"
    shutil.copy2(cfg["csv"], backup)

    pd.concat([existing, pd.DataFrame(rows)], ignore_index=True).to_csv(cfg["csv"], index=False)
    print(f"  added {len(rows):,} rows (backup: {os.path.basename(backup)})")


=== 14-class labels -> classification_results_paip.csv
  training_data_19_nonMSIH: 898 patches


  classifying:   0%|          | 0/15 [00:00<?, ?it/s]c:\Users\datainsight\anaconda3\envs\exaonepath\lib\site-packages\timm\layers\attention.py:80: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  x = F.scaled_dot_product_attention(


  training_data_30_MSIH: 1,466 patches


  training_data_41_nonMSIH: 170 patches


  training_data_42_MSIH: 1,116 patches


...

  training_data_30_MSIH: 1,466 patches


  training_data_41_nonMSIH: 170 patches


  training_data_42_MSIH: 1,116 patches


  training_data_46_nonMSIH: 569 patches


  added 4,219 rows (backup: classification_results_paip_15_classes.csv.bak-20260810-015136)


### 3b. Aggregate by label

For each slide: average the 5 crops of every patch, group the patches by the label CONCH gave them, and
average within each group. One row per class, in the config's class order. Classes with no patches stay zero.

In [5]:
def caption_aggregate(model_name, slide, labels_by_patch, classes):
    """Group the slide's patches by predicted class and average each group."""
    cfg = MODELS[model_name]
    groups = {c: [] for c in classes}

    for patch_name, label in labels_by_patch.items():
        path = os.path.join(feature_dir(model_name, slide), patch_name.replace(".png", ".pt"))
        if not os.path.exists(path):
            continue

        t = torch.load(path, map_location="cpu", weights_only=True)   # (5, D)
        groups[label].append(t.mean(dim=0))                           # (D,)

    rows = []
    for c in classes:
        if groups[c]:
            rows.append(torch.stack(groups[c]).mean(dim=0))
        else:
            rows.append(torch.zeros(cfg["dim"]))

    return torch.stack(rows)          # (n_classes, D)


for n_classes, cfg in LABEL_SETS.items():
    print(f"\n=== {n_classes}-class aggregation")
    df = pd.read_csv(cfg["csv"])
    df["slide"] = df["patch_name"].str.split("_").str[:4].str.join("_")
    df = df[df["slide"].isin(SLIDES)]

    for model_name in MODELS:
        out_dir = os.path.join(AGG_ROOT, cfg["agg_dir"], model_name)
        os.makedirs(out_dir, exist_ok=True)

        for slide in SLIDES:
            sub = df[df["slide"] == slide]
            if sub.empty:
                print(f"  {model_name} / {slide}: no labels, run 3a first")
                continue

            labels_by_patch = dict(zip(sub["patch_name"], sub["label"]))
            tensor = caption_aggregate(model_name, slide, labels_by_patch, cfg["classes"])

            used = int((tensor.abs().sum(dim=1) > 0).sum())
            torch.save(tensor, os.path.join(out_dir, slide + ".pt"))
            print(f"  {model_name:<14} {slide:<26} {tuple(tensor.shape)}  "
                  f"{used}/{n_classes} classes present")


=== 14-class aggregation
  ConchV1        training_data_19_nonMSIH   (14, 512)  12/14 classes present
  ConchV1        training_data_30_MSIH      (14, 512)  12/14 classes present
  ConchV1        training_data_41_nonMSIH   (14, 512)  10/14 classes present
  ConchV1        training_data_42_MSIH      (14, 512)  13/14 classes present
  ConchV1        training_data_46_nonMSIH   (14, 512)  12/14 classes present
  Conch1_5       training_data_19_nonMSIH   (14, 768)  12/14 classes present
  Conch1_5       training_data_30_MSIH      (14, 768)  12/14 classes present
  Conch1_5       training_data_41_nonMSIH   (14, 768)  10/14 classes present
  Conch1_5       training_data_42_MSIH      (14, 768)  13/14 classes present
  Conch1_5       training_data_46_nonMSIH   (14, 768)  12/14 classes present
  UNI2           training_data_19_nonMSIH   (14, 1536)  12/14 classes present
  UNI2           training_data_30_MSIH      (14, 1536)  12/14 classes present
  UNI2           training_data_41_nonMSIH   (14,

---
## 4. Check

Every folder should now hold 78 files, and the five new tensors should match the shape of the 73 already
there.

In [6]:
CHECKS = [("Averaging", None)] + \
         [("Tissue_Type_Clustering", 9)] + \
         [(LABEL_SETS[14]["agg_dir"], 14), (LABEL_SETS[15]["agg_dir"], 15)]

for method, n_rows in CHECKS:
    print(f"\n{method}")

    for model_name in MODELS:
        out_dir = os.path.join(AGG_ROOT, method, model_name)
        if not os.path.isdir(out_dir):
            print(f"  {model_name:<14} folder missing")
            continue

        files = [f for f in os.listdir(out_dir) if f.endswith(".pt")]

        # shape of a slide that was already done
        old = next(f for f in sorted(files) if f[:-3] not in SLIDES)
        ref = tuple(torch.load(os.path.join(out_dir, old), map_location="cpu",
                               weights_only=True).shape)

        problems = []
        for slide in SLIDES:
            path = os.path.join(out_dir, slide + ".pt")
            if not os.path.exists(path):
                problems.append(f"{slide} missing")
                continue

            shape = tuple(torch.load(path, map_location="cpu", weights_only=True).shape)
            if shape != ref:
                problems.append(f"{slide} is {shape}, expected {ref}")

        status = "ok" if not problems else " | ".join(problems)
        print(f"  {model_name:<14} {len(files):>3} files  ref {str(ref):<12} {status}")


Averaging
  ConchV1         78 files  ref (1, 512)     ok
  Conch1_5        78 files  ref (1, 768)     ok
  UNI2            78 files  ref (1, 1536)    ok
  Virchow2        78 files  ref (1, 2560)    ok
  H-Optimus-1     78 files  ref (1, 1536)    ok

Tissue_Type_Clustering
  ConchV1         78 files  ref (9, 512)     ok
  Conch1_5        78 files  ref (9, 768)     ok
  UNI2            78 files  ref (9, 1536)    ok
  Virchow2        78 files  ref (9, 2560)    ok
  H-Optimus-1     78 files  ref (9, 1536)    ok

Caption_based_aggregation
  ConchV1         78 files  ref (14, 512)    ok
  Conch1_5        78 files  ref (14, 768)    ok
  UNI2            78 files  ref (14, 1536)   ok
  Virchow2        78 files  ref (14, 2560)   ok
  H-Optimus-1     78 files  ref (14, 1536)   ok

Caption_based_aggregation_15_classes
  ConchV1         78 files  ref (15, 512)    ok
  Conch1_5        78 files  ref (15, 768)    ok
  UNI2            78 files  ref (15, 1536)   ok
  Virchow2        78 files  ref (15,

---
# 5. TITAN

TITAN is not an averaging rule like the three above — it is a **pretrained slide encoder**. You hand it every
patch of a slide plus where each patch sits, and it returns one `(1, 768)` embedding.

Two things make it different from sections 1–3:

- **It does not read the features we extracted.** TITAN ships its own CONCH v1.5 tile encoder
  (`titan.return_conch()`), and re-encodes the patch PNGs with it. That is why `slide_aggregation\TITAN` has
  only a `Conch1_5` folder — it is TITAN end to end, not "TITAN applied to our CONCH features".
- **It is spatially aware.** Along with the features it takes each patch's `(x, y)` and the level-0 patch
  size, so the slide is treated as a map rather than a bag of patches. The coordinates come out of the patch
  filenames, and the patches are sorted by position before encoding.

`patch_size_level0` comes from the slide's objective power: 1024 at 40x, 512 at 20x. All 78 PAIP slides are
40x, so all five here use 1024.

This section loads TITAN, which is a separate multi-GB model from the CONCH used in section 3a. If the GPU is
tight, restart the kernel, run the config cell, then come straight here.

In [8]:
import re
import h5py
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoModel
from huggingface_hub import login

TITAN_OUT  = os.path.join(AGG_ROOT, "TITAN", "Conch1_5")
H5_DIR     = os.path.join(PAIP_ROOT, "slide_h5_files", "TITAN")
SLIDE_META = os.path.join(REPO_ROOT, "data_preprocessing", "slides_metadata",
                          "paip_slide_metadata_filtered.csv")

BATCH_SIZE_TITAN = 64


class Titan:
    """TITAN plus the CONCH v1.5 tile encoder that comes with it."""

    def __init__(self):
        token = os.getenv("HF_TOKEN")
        if token:
            login(token=token)

        print("loading TITAN...")
        self.titan = AutoModel.from_pretrained("MahmoodLab/TITAN", trust_remote_code=True)
        self.conch, self.eval_transform = self.titan.return_conch()

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.titan = self.titan.to(self.device).eval()
        self.conch = self.conch.to(self.device).eval()
        print("ready on", self.device)


def patch_coords(path):
    """Pull (x, y) out of a patch filename - works for both naming styles."""
    m = re.search(r"_x(\d+)_y(\d+)", os.path.basename(path))
    if not m:
        raise ValueError(f"no coordinates in {path}")
    return int(m.group(1)), int(m.group(2))


def load_batch(paths, transform):
    """Load and transform a batch of patch images."""
    def load_one(p):
        try:
            return transform(Image.open(p).convert("RGB"))
        except Exception as e:
            print(f"  couldn't load {p}: {e}")
            return None

    with ThreadPoolExecutor(max_workers=8) as pool:
        tensors = [t for t in pool.map(load_one, paths) if t is not None]

    return torch.stack(tensors) if tensors else torch.empty(0)


def write_h5(features, coords, slide, patch_size_level0):
    """Same H5 layout as the original TITAN notebook - kept for the record."""
    os.makedirs(H5_DIR, exist_ok=True)
    out = os.path.join(H5_DIR, f"{slide}_patch_features.h5")

    with h5py.File(out, "w") as f:
        f.create_dataset("features", data=features[np.newaxis, :].astype(np.float32),
                         dtype=np.float32)
        ds = f.create_dataset("coords", data=coords[np.newaxis, :].astype(np.int64),
                              dtype=np.int64)

        for key, value in {
            "contour_fn": "four_pt",
            "custom_downsample": 2.0,
            "downsample": np.array([1., 1.]),
            "name": slide,
            "patch_level": 0,
            "patch_size": patch_size_level0,
            "patch_size_level0": patch_size_level0,
            "step_size": patch_size_level0,
            "use_padding": True,
        }.items():
            ds.attrs[key] = value

    return out

In [9]:
# objective power -> level-0 patch size (all PAIP slides are 40x)
meta = pd.read_csv(SLIDE_META)
power = dict(zip(meta["renamed"].str.lower(), meta["Objective Power"]))

for slide in SLIDES:
    p = power.get(slide.lower())
    print(f"{slide:<26} {p}x -> patch_size_level0 = {1024 if p == 40 else 512}")

training_data_19_nonMSIH   40x -> patch_size_level0 = 1024
training_data_30_MSIH      40x -> patch_size_level0 = 1024
training_data_41_nonMSIH   40x -> patch_size_level0 = 1024
training_data_42_MSIH      40x -> patch_size_level0 = 1024
training_data_46_nonMSIH   40x -> patch_size_level0 = 1024


In [10]:
titan = Titan()

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful
loading TITAN...
ready on cuda


In [11]:
os.makedirs(TITAN_OUT, exist_ok=True)
valid_names = set(pd.read_csv(NON_WHITE_CSV)["patch_name"].astype(str))

for slide in SLIDES:
    out_path = os.path.join(TITAN_OUT, slide + ".pt")
    if os.path.exists(out_path):
        print(f"{slide}: already done")
        continue

    obj_power = power.get(slide.lower())
    if obj_power not in (20, 40):
        print(f"{slide}: objective power {obj_power} not supported, skipping")
        continue
    patch_size_level0 = 1024 if obj_power == 40 else 512

    patches = [p for p in glob.glob(os.path.join(PATCH_DATA_DIR, slide, "*.png"))
               if os.path.basename(p) in valid_names]
    if not patches:
        print(f"{slide}: no non-white patches, skipping")
        continue

    # TITAN cares about position, so feed the patches in spatial order
    patches = sorted(patches, key=patch_coords)
    print(f"\n{slide}: {len(patches):,} patches at {obj_power}x")

    feats = []
    with torch.inference_mode():
        for i in tqdm(range(0, len(patches), BATCH_SIZE_TITAN), desc="  encoding", leave=False):
            batch = load_batch(patches[i:i + BATCH_SIZE_TITAN], titan.eval_transform)
            if batch.nelement() == 0:
                continue
            feats.append(titan.conch(batch.to(titan.device)).cpu().numpy())

    feats = np.concatenate(feats, axis=0)                    # (N, 768)
    coords = np.array([patch_coords(p) for p in patches], dtype=np.int64)   # (N, 2)
    print(f"  features {feats.shape}, coords {coords.shape}")

    h5_path = write_h5(feats, coords, slide, patch_size_level0)

    with torch.inference_mode():
        slide_emb = titan.titan.encode_slide_from_patch_features(
            torch.from_numpy(feats).unsqueeze(0).to(titan.device),
            torch.from_numpy(coords).unsqueeze(0).to(titan.device),
            patch_size_level0,
        )

    torch.save(slide_emb.cpu(), out_path)
    print(f"  slide embedding {tuple(slide_emb.shape)} -> {out_path}")


training_data_19_nonMSIH: 898 patches at 40x


  features (898, 768), coords (898, 2)
  slide embedding (1, 768) -> D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\TITAN\Conch1_5\training_data_19_nonMSIH.pt

training_data_30_MSIH: 1,466 patches at 40x


  features (1466, 768), coords (1466, 2)
  slide embedding (1, 768) -> D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\TITAN\Conch1_5\training_data_30_MSIH.pt

training_data_41_nonMSIH: 170 patches at 40x


  features (170, 768), coords (170, 2)
  slide embedding (1, 768) -> D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\TITAN\Conch1_5\training_data_41_nonMSIH.pt

training_data_42_MSIH: 1,116 patches at 40x


  features (1116, 768), coords (1116, 2)
  slide embedding (1, 768) -> D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\TITAN\Conch1_5\training_data_42_MSIH.pt

training_data_46_nonMSIH: 569 patches at 40x


  features (569, 768), coords (569, 2)
  slide embedding (1, 768) -> D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\TITAN\Conch1_5\training_data_46_nonMSIH.pt


### Check TITAN

`TITAN\Conch1_5` should now hold 78 files, and the five new embeddings should match the shape of the 73.

In [12]:
files = [f for f in os.listdir(TITAN_OUT) if f.endswith(".pt")]

old = next(f for f in sorted(files) if f[:-3] not in SLIDES)
ref = tuple(torch.load(os.path.join(TITAN_OUT, old), map_location="cpu", weights_only=True).shape)
print(f"{len(files)} files | reference ({old[:-3]}) {ref}\n")

for slide in SLIDES:
    path = os.path.join(TITAN_OUT, slide + ".pt")
    if not os.path.exists(path):
        print(f"  {slide:<26} MISSING")
        continue

    t = torch.load(path, map_location="cpu", weights_only=True)
    h5 = os.path.join(H5_DIR, f"{slide}_patch_features.h5")
    ok = tuple(t.shape) == ref and torch.isfinite(t).all()
    print(f"  {slide:<26} {tuple(t.shape)}  h5 {'yes' if os.path.exists(h5) else 'no ':<3} "
          f"{'ok' if ok else 'CHECK'}")

78 files | reference (training_data_01_MSIH) (1, 768)

  training_data_19_nonMSIH   (1, 768)  h5 yes ok
  training_data_30_MSIH      (1, 768)  h5 yes ok
  training_data_41_nonMSIH   (1, 768)  h5 yes ok
  training_data_42_MSIH      (1, 768)  h5 yes ok
  training_data_46_nonMSIH   (1, 768)  h5 yes ok
